# Japon : vingt ans de nominal plat, des volumes en hausse · *Japan: twenty years of flat nominal, rising volumes*

Notebook compagnon du chapitre **10. PIB réel et PIB nominal : pourquoi corriger de l'inflation** — [lire l'article](https://nmlab.io/ressources/pib-reel-pib-nominal).
Companion notebook to chapter **10. Real GDP and Nominal GDP: Why Correct for Inflation** — [read the article](https://nmlab.io/en/ressources/real-gdp-nominal-gdp).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_japan() -> tuple[Series, Series]:
    """PIB nominal (JPNNGDP) et réel (JPNRGDPEXP) du Japon, trimestriels, en direct depuis FRED,
    ramenés en base 100 = moyenne 1995.
    Japan nominal and real GDP, quarterly, live from FRED, indexed to 100 = 1995 average."""
    nominal = nm.load_fred("JPNNGDP")
    real = nm.load_fred("JPNRGDPEXP")
    nominal = nominal / nominal[nominal.index.year == 1995].mean() * 100
    real = real / real[real.index.year == 1995].mean() * 100
    return nominal, real

japan_nominal, japan_real = load_japan()


import pandas as pd
import matplotlib.dates as mdates
from matplotlib.figure import Figure

LABELS = {
    "fr": dict(
        title="Japon : vingt ans de nominal plat, des volumes en hausse",
        sub="PIB nominal et PIB réel du Japon, base 100 = moyenne 1995, données trimestrielles",
        ylab="indice, base 100 en 1995",
        legend_nom="PIB nominal", legend_real="PIB réel",
        base_line="niveau de 1995 = 100",
        ann_real="2015 : réel +18,5 %",
        ann_nom="2015 : nominal +3,5 %\ndepuis 1995",
        ann_2022="2022- : l'inflation revient,\nle nominal décolle",
        note="La déflation inverse l'illusion : le nominal sous-estime l'activité. En 2025, le niveau des prix\n"
             "du PIB japonais dépasse celui de 1995 de moins de 1 %. Source : Cabinet Office via FRED."),
    "en": dict(
        title="Japan: twenty years of flat nominal, rising volumes",
        sub="Japan's nominal and real GDP, indexed to 100 = 1995 average, quarterly data",
        ylab="index, 1995 = 100",
        legend_nom="nominal GDP", legend_real="real GDP",
        base_line="1995 level = 100",
        ann_real="2015: real +18.5%",
        ann_nom="2015: nominal +3.5%\nsince 1995",
        ann_2022="2022-: inflation returns,\nthe nominal takes off",
        note="Deflation flips the illusion: the nominal understates activity. In 2025, Japan's GDP price level\n"
             "stands less than 1% above its 1995 level. Source: Cabinet Office via FRED."),
}

def _at(series: Series, year: int, quarter_index: int = 0):
    """(date, valeur) d'un trimestre d'une année donnée."""
    mask = series.index.year == year
    return series.index[mask][quarter_index], float(series[mask].iloc[quarter_index])

def build_figure(nominal: Series, real: Series, lang: str) -> Figure:
    """Deux courbes trimestrielles en base 100 (1995) : nominal (bleu) plat, réel (rose) en hausse."""
    text = LABELS[lang]
    fig = nm.figure(height_px=1140)
    ax = nm.axes(fig, left=0.072)
    ax.axhline(100, color=nm.COLORS["muted"], linestyle="--", lw=1.6, zorder=1)
    line_real, = ax.plot(real.index, real, color=nm.COLORS["rose"], lw=2.6, label=text["legend_real"], zorder=3)
    line_nom, = ax.plot(nominal.index, nominal, color=nm.COLORS["blue"], lw=2.6, label=text["legend_nom"], zorder=4)
    ax.set_ylim(87, 133)
    ax.set_yticks(range(90, 131, 10))
    ax.set_ylabel(text["ylab"])
    ax.set_xlim(pd.Timestamp("1993-06-01"), pd.Timestamp("2026-09-01"))
    ax.xaxis.set_major_locator(mdates.YearLocator(5))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.text(pd.Timestamp("2007-06-01"), 101.4, text["base_line"], ha="left", va="bottom",
            fontsize=20, color=nm.COLORS["muted"])

    d_real, v_real = _at(real, 2015)
    ax.annotate(text["ann_real"], xy=(d_real, v_real), xytext=(pd.Timestamp("2004-01-01"), 124),
                ha="left", va="center", fontsize=22, color=nm.COLORS["rose"],
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["rose"], lw=1.4))
    d_nom, v_nom = _at(nominal, 2015)
    ax.annotate(text["ann_nom"], xy=(d_nom, v_nom), xytext=(pd.Timestamp("2007-06-01"), 90),
                ha="center", va="center", fontsize=22, color=nm.COLORS["blue"], linespacing=1.5,
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["blue"], lw=1.4))
    d_22, v_22 = _at(nominal, 2023)
    ax.annotate(text["ann_2022"], xy=(d_22, v_22), xytext=(pd.Timestamp("2015-06-01"), 130),
                ha="left", va="center", fontsize=22, color=nm.COLORS["blue"], linespacing=1.5,
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["blue"], lw=1.4))

    ax.legend(handles=[line_nom, line_real], loc="upper left", frameon=False, fontsize=23,
              labelcolor=nm.COLORS["text"], handlelength=1.6, borderaxespad=1.0)
    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig

build_figure(japan_nominal, japan_real, LANG)